In [1]:
import pandas as pd
from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.metrics import AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import nest_asyncio

/Users/cberches/Documents/GitHub/AttritionTest/capstone_env/lib/python3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/cberches/Documents/GitHub/AttritionTest/capstone_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/js/rvl93zld41n502z8rv_gr2qw0000gq/T/ipykernel_63040/603833031.py:4: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall,

In [3]:
from agent import policy_retriever, rag_node

# Initialize LLM and Embeddings for Evaluation
eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
eval_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# 1. Load your golden test set dataset (CSV with 'question', 'ground_truth')
test_df = pd.read_csv('hr_testset_improved.csv')

samples = []

for _, row in test_df.iterrows():
    question = row['question']
    ground_truth = row['ground_truth']
    
    # 2. Run the question through the imported RAG retriever logic
    retrieved_docs = policy_retriever.invoke(question)
    contexts = [doc.page_content for doc in retrieved_docs]
    
    # Define a simple evaluation prompt
    eval_prompt = f"""You are a professional HR Business Partner. 
        Answer the user's question based strictly on the provided HR Playbook context.
        
        Context:
        {"\n".join(contexts)}
        
        Question: {question}
    """
    
    # Invoke the LLM with the prompt string
    response = eval_llm.invoke(eval_prompt)
    answer = response.content
    
    # 3. Format as a SingleTurnSample for RAGAS
    samples.append(
        SingleTurnSample(
            user_input=question,
            response=answer,
            retrieved_contexts=contexts,
            reference=ground_truth,
        )
    )

# 4. Create Evaluation Dataset and Run RAGAS Evaluation
eval_dataset = EvaluationDataset(samples=samples)
metrics = [Faithfulness(), AnswerRelevancy(), ContextRecall(), ContextPrecision(), AnswerCorrectness()]

results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=eval_llm,
    embeddings=eval_embeddings,
)

# 5. Output the final DataFrame containing the scores
results_df = results.to_pandas()
print(results_df)

SyntaxError: f-string expression part cannot include a backslash (1520533685.py, line 21)

In [ ]:
results

{'faithfulness': 0.0000, 'answer_relevancy': 0.3887, 'context_recall': 0.0000, 'context_precision': 0.0000, 'answer_correctness': 0.1661}

In [ ]:

results.to_pandas()

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall,context_precision,answer_correctness
0,What is the cash bonus for the Bayanihan Loyal...,[],The policy does not cover the cash bonus for t...,"Upon reaching 10 Total Working Years, employee...",0.0,0.868974,0.0,0.0,0.178201
1,How many days of paid sabbatical leave are emp...,[],The policy does not cover it.,Employees surpassing 10 Total Working Years ar...,0.0,0.229789,0.0,0.0,0.034483
2,What are the eligibility requirements for an e...,[],The policy does not cover the eligibility requ...,Employees with over 5 Total Working Years auto...,0.0,0.890538,0.0,0.0,0.184692
3,What is the baseline minimum Percent Salary Hi...,[],The policy does not cover the baseline minimum...,All non-PIP employees must receive a baseline ...,0.0,0.775612,0.0,0.0,0.627178
4,What percentage range of Percent Salary Hike s...,[],The policy does not cover the percentage range...,High-flight-risk employees with an Outstanding...,0.0,0.000000,0.0,0.0,0.196821
...,...,...,...,...,...,...,...,...,...
95,What is the percentage weight assigned to Tota...,[],The policy does not cover the percentage weigh...,18.88%.,0.0,0.901085,0.0,0.0,0.077155
96,What is the percentage weight assigned to Perc...,[],The policy does not cover the percentage weigh...,6.53%.,0.0,0.904315,0.0,0.0,0.079895
97,What is the percentage weight assigned to Year...,[],The policy does not cover the percentage weigh...,5.55%.,0.0,0.916863,0.0,0.0,0.075854
98,What is the percentage weight assigned to Trai...,[],The policy does not cover the percentage weigh...,3.28%.,0.0,0.904017,0.0,0.0,0.076255
